In [1]:
import pandas as pd
import re
import string
import nltk
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [2]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to E:\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to E:\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
data = pd.read_csv("../../dataset/politik_merge.csv")
data

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source
0,Jokowi Kenakan Pakaian Adat Betawi di Sidang T...,16/08/2024,https://nasional.kompas.com/read/2024/08/16/11...,"JAKARTA, KOMPAS.com - Presiden Joko Widodo me...",Presiden Jokowi,Jokowi,sidang tahunan MPR RI 2024,Jokowi adat Betawi sidang mpr 2024,Megawati tak hadiri sidang tahunan MPR 2024,kompas
1,Amnesty International Beberkan 6 Indikator Kri...,2024-07-18,https://nasional.tempo.co/read/1893144/amnesty...,"TEMPO.CO, Jakarta - Amnesty International Indo...",Amnesty International,Amnesty International Indonesia,Kebebasan Berpendapat,Indeks Demokrasi,Revisi UU TNI,tempo
2,"Jelang Long Weekend, Stasiun Kereta Cepat Hali...","Rabu, 08 Mei 2024 19:18 WIB",https://news.detik.com/berita/d-7331666/jelang...,"Stasiun kereta cepat Whoosh di Halim, Jakarta ...",kereta cepat whoosh,stasiun halim,long weekend,NaN,NaN,detik
3,KPU Tegaskan Pemilih Tak Terdaftar di DPT Bisa...,13/02/2024,https://nasional.kompas.com/read/2024/02/13/21...,"JAKARTA, KOMPAS.com - Komisi Pemilihan Umum (...",KPU,pemilu 2024,Hasyim Asy'ari,NaN,NaN,kompas
4,Kemenag Luncurkan Gerakan Senam Haji Jaga Keta...,2024-04-29,https://nasional.tempo.co/read/1861810/kemenag...,"TEMPO.CO, Jakarta - Kementerian Agama atau Kem...",Senam Haji,Kemenag,Jemaah Haji,Ibadah Haji,Asrama Haji,tempo
...,...,...,...,...,...,...,...,...,...,...
45776,"Terima Mohammad Shtayyeh, Bamsoet: RI Tegas Du...","Minggu, 25 Agu 2024 15:24 WIB",https://news.detik.com/berita/d-7507663/terima...,Ketua MPR RI ke-16 Bambang Soesatyo menegaskan...,mpr,bamsoet,palestina,mohammad shtayyeh,NaN,detik
45777,Kegiatan Setelah Kalah Pilpres: Anies Jeda Pol...,2024-04-30,https://nasional.tempo.co/read/1862571/kegiata...,"TEMPO.CO, Jakarta - Mantan calon presiden nomo...",Anies,Ganjar,Mahfud MD,Pilpres,KAGAMA,tempo
45778,"Prabowo Bicara Malin Kundang, Hasto PDIP: Blunder",2024-01-14,https://nasional.tempo.co/read/1821140/prabowo...,"TEMPO.CO, Jakarta - Sekretaris Jenderal Partai...",PDIP,Prabowo,Malin Kundang,Hasto Kristiyanto,Jokowi,tempo
45779,"Gugatannya Disebut Cengeng dan Cacat Formil, T...",2024-03-27,https://nasional.tempo.co/read/1850093/gugatan...,"TEMPO.CO, Jakarta - Anggota dari Tim Pembela P...",Timnas Amin,Gugatan Pilpres,Prabowo-Gibran,Hotman Paris Hutapea,Otto Hasibuan,tempo


In [4]:
df = data[['Judul','Content']].head(100)
df

,Judul,Content
0,Jokowi Kenakan Pakaian Adat Betawi di Sidang T...,"JAKARTA, KOMPAS.com - Presiden Joko Widodo me..."
1,Amnesty International Beberkan 6 Indikator Kri...,"TEMPO.CO, Jakarta - Amnesty International Indo..."
2,"Jelang Long Weekend, Stasiun Kereta Cepat Hali...","Stasiun kereta cepat Whoosh di Halim, Jakarta ..."
3,KPU Tegaskan Pemilih Tak Terdaftar di DPT Bisa...,"JAKARTA, KOMPAS.com - Komisi Pemilihan Umum (..."
4,Kemenag Luncurkan Gerakan Senam Haji Jaga Keta...,"TEMPO.CO, Jakarta - Kementerian Agama atau Kem..."
...,...,...
95,Ini 5 Program Prioritas Kemendikbud untuk Guru...,"TEMPO.CO, Riau - Kebijakan Merdeka Belajar sa..."
96,Cerita Korban Penipuan Visa Haji Ilegal: Panas...,"MEKKAH, KOMPAS.com- Lukman (bukan nama sebena..."
97,"Gerindra Sebut ""Reshuffle"" Menkumham untuk Sin...","JAKARTA, KOMPAS.com - Ketua Harian Partai Ger..."
98,Lawrence Wong Dilantik Jadi PM Singapura,Lawrence Wong dilantik sebagai Perdana Menteri...


In [5]:
def casefolding(text):
    if pd.isnull(text):
        return ""
    
    return text.lower()

# df['Judul'] = df['Judul'].apply(casefolding)
df['Content'] = df['Content'].apply(casefolding)

df['Content'].head(10)

0     jakarta, kompas.com - presiden joko widodo me...
1    tempo.co, jakarta - amnesty international indo...
2    stasiun kereta cepat whoosh di halim, jakarta ...
3     jakarta, kompas.com - komisi pemilihan umum (...
4    tempo.co, jakarta - kementerian agama atau kem...
5    tempo.co, jakarta - anggota badan pengawas pem...
6    polisi menyebutkan pria inisial dk alias depoy...
7    satu unit truk fuso mogok di pelintasan kereta...
8    tempo.co, jakarta - bendahara umum pdip, olly ...
9     tangerang selatan, kompas.com - seutas tali b...
Name: Content, dtype: object

In [6]:
# Fungsi tokenisasi
def tokenisasi(text):
    if pd.isnull(text):
        return []
    
    return word_tokenize(text)

# Menerapkan tokenisasi pada kolom Judul dan Content
# df['Judul'] = df['Judul'].apply(tokenisasi)
df['Content'] = df['Content'].apply(tokenisasi)

# Menampilkan hasil
df['Content'].head(10)

0    [jakarta, ,, kompas.com, -, presiden, joko, wi...
1    [tempo.co, ,, jakarta, -, amnesty, internation...
2    [stasiun, kereta, cepat, whoosh, di, halim, ,,...
3    [jakarta, ,, kompas.com, -, komisi, pemilihan,...
4    [tempo.co, ,, jakarta, -, kementerian, agama, ...
5    [tempo.co, ,, jakarta, -, anggota, badan, peng...
6    [polisi, menyebutkan, pria, inisial, dk, alias...
7    [satu, unit, truk, fuso, mogok, di, pelintasan...
8    [tempo.co, ,, jakarta, -, bendahara, umum, pdi...
9    [tangerang, selatan, ,, kompas.com, -, seutas,...
Name: Content, dtype: object

In [7]:
# Membuat stopword remover
factory = StopWordRemoverFactory()
stopwords = factory.get_stop_words()

# Fungsi stopword removal
def stopword_removal(tokens):
    return [word for word in tokens if word not in stopwords]

# Menerapkan stopword removal pada kolom Judul dan Content
# df['Judul'] = df['Judul'].apply(stopword_removal)
df['Content'] = df['Content'].apply(stopword_removal)

# Menampilkan hasil
df['Content'].head(10)

0    [jakarta, ,, kompas.com, -, presiden, joko, wi...
1    [tempo.co, ,, jakarta, -, amnesty, internation...
2    [stasiun, kereta, cepat, whoosh, halim, ,, jak...
3    [jakarta, ,, kompas.com, -, komisi, pemilihan,...
4    [tempo.co, ,, jakarta, -, kementerian, agama, ...
5    [tempo.co, ,, jakarta, -, anggota, badan, peng...
6    [polisi, menyebutkan, pria, inisial, dk, alias...
7    [satu, unit, truk, fuso, mogok, pelintasan, ke...
8    [tempo.co, ,, jakarta, -, bendahara, umum, pdi...
9    [tangerang, selatan, ,, kompas.com, -, seutas,...
Name: Content, dtype: object

In [8]:
# Membuat stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Fungsi stemming
def stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

# Menerapkan stemming pada kolom Judul dan Content
# df['Judul'] = df['Judul'].apply(stemming)
df['Content'] = df['Content'].apply(stemming)

# Menampilkan hasil
df['Content'].head(10)

KeyboardInterrupt: 